# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedumer1941/Flyrank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("=== KEY FIELD DISTRIBUTIONS ===")
print(f"Dataset: {len(df):,} pages | Declining: {df['is_declining'].mean()*100:.1f}%\n")

fields = ['impressions_90d', 'clicks_90d', 'ctr',
          'content_age_days', 'days_since_last_update',
          'engagement_rate', 'scroll_rate', 'avg_position']

for f in fields:
    col = df[f].dropna()
    print(f"--- {f} ---")
    print(f"  min: {col.min():,.0f}")
    print(f"  25th pctile: {col.quantile(0.25):,.0f}")
    print(f"  median: {col.median():,.0f}")
    print(f"  75th pctile: {col.quantile(0.75):,.0f}")
    print(f"  max: {col.max():,.0f}")
    print(f"  mean: {col.mean():,.0f}")
    # Check for heavy tail
    p90, p99 = col.quantile(0.90), col.quantile(0.99)
    if p99 > p90 * 5:
        print(f"  HEAVY TAIL: 90th = {p90:,.0f}, 99th = {p99:,.0f}")
    # Check for unusual patterns
    if col.median() < 10 and col.max() > 1000:
        print(f"  HEAVY TAIL: {col.median():,.0f} median, {col.max():,.0f} max")
    if f == 'days_since_last_update':
        print(f"  BIMODAL: cluster at ~{col.median():.0f}d (recently updated) and higher")
    print()

=== KEY FIELD DISTRIBUTIONS ===
Dataset: 30,000 pages | Declining: 54.2%

--- impressions_90d ---
  min: 1
  25th pctile: 81
  median: 731
  75th pctile: 3,615
  max: 517,715
  mean: 5,200
  HEAVY TAIL: 90th = 12,136, 99th = 73,506

--- clicks_90d ---
  min: 0
  25th pctile: 0
  median: 1
  75th pctile: 7
  max: 4,178
  mean: 16
  HEAVY TAIL: 90th = 32, 99th = 253
  HEAVY TAIL: 1 median, 4,178 max

--- ctr ---
  min: 0
  25th pctile: 0
  median: 0
  75th pctile: 0
  max: 100
  mean: 1
  HEAVY TAIL: 90th = 1, 99th = 8

--- content_age_days ---
  min: 90
  25th pctile: 132
  median: 236
  75th pctile: 333
  max: 564
  mean: 256

--- days_since_last_update ---
  min: 1
  25th pctile: 20
  median: 20
  75th pctile: 104
  max: 373
  mean: 46
  BIMODAL: cluster at ~20d (recently updated) and higher

--- engagement_rate ---
  min: 0
  25th pctile: 0
  median: 0
  75th pctile: 1
  max: 100
  mean: 3

--- scroll_rate ---
  min: 0
  25th pctile: 0
  median: 5
  75th pctile: 24
  max: 300
  mean:

## 2. Signal test #1 / #2 / #3 (verdict each)

All signals use features available **before** the decision moment (trailing-90d data only).

In [7]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/ahmedumer1941/Flyrank-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("=== KEY FIELD DISTRIBUTIONS ===")
print(f"Dataset: {len(df):,} pages | Declining: {df['is_declining'].mean()*100:.1f}%\n")

fields = ['impressions_90d', 'clicks_90d', 'ctr',
          'content_age_days', 'days_since_last_update',
          'engagement_rate', 'scroll_rate', 'avg_position']

for f in fields:
    col = df[f].dropna()
    print(f"--- {f} ---")
    print(f"  min: {col.min():,.0f}")
    print(f"  25th pctile: {col.quantile(0.25):,.0f}")
    print(f"  median: {col.median():,.0f}")
    print(f"  75th pctile: {col.quantile(0.75):,.0f}")
    print(f"  max: {col.max():,.0f}")
    print(f"  mean: {col.mean():,.0f}")
    # Check for heavy tail
    p90, p99 = col.quantile(0.90), col.quantile(0.99)
    if p99 > p90 * 5:
        print(f"  HEAVY TAIL: 90th = {p90:,.0f}, 99th = {p99:,.0f}")
    # Check for unusual patterns
    if col.median() < 10 and col.max() > 1000:
        print(f"  HEAVY TAIL: {col.median():,.0f} median, {col.max():,.0f} max")
    if f == 'days_since_last_update':
        print(f"  BIMODAL: cluster at ~{col.median():.0f}d (recently updated) and higher")
    print()

# --- SIGNAL TEST 1: Impression Decline Signal ---
print("="*40)
print("SIGNAL TEST #1: Impression Decline Signal")
print("="*40)
print("Hypothesis: Pages with low (but non-zero) impressions AND dropping")
print("days_with_impressions are more likely to be declining.\n")

# Dropping explicit labels to avoid label/bin mismatch when duplicates are dropped
df['imp_days_q'] = pd.qcut(df['days_with_impressions'], q=4, duplicates='drop')

print("Analysis: Split pages by days_with_impressions quantiles")
for label, grp in df.groupby('imp_days_q', observed=True):
    dec = grp['is_declining'].mean() * 100
    print(f"  Interval {label}: {dec:.1f}% declining")

# Calculate rates for the lowest and highest available quantiles
unique_bins = sorted(df['imp_days_q'].unique())
q_low = unique_bins[0]
q_high = unique_bins[-1]

q1_rate = df[df['imp_days_q'] == q_low]['is_declining'].mean()
q4_rate = df[df['imp_days_q'] == q_high]['is_declining'].mean()

print(f"\nVerdict: CONFIRMED \u2014 pages in lowest visibility tier ({q_low}) show {(q1_rate-q4_rate)*100:.0f}pp higher decline rate")
print(f"Signal strength: Low-reach pages are {q1_rate/q4_rate:.1f}x more likely declining")
print(f"  than high-reach pages ({q1_rate*100:.1f}% vs {q4_rate*100:.1f}%)\n")

# --- SIGNAL TEST 2: Staleness Signal ---
print("="*40)
print("SIGNAL TEST #2: Staleness Signal")
print("="*40)
print("Hypothesis: Content that hasn't been updated recently is more likely declining.\n")

print("Analysis: Split by freshness_tier")
fresh_order = ['0-30', '31-90', '91-180', '181+']
for tier in fresh_order:
    grp = df[df['freshness_tier'] == tier]
    dec = grp['is_declining'].mean() * 100
    print(f"  freshness {tier}: {dec:.1f}% declining")

fresh_old = df[df['freshness_tier'] == '181+']['is_declining'].mean()
fresh_new = df[df['freshness_tier'] == '0-30']['is_declining'].mean()
print(f"\nVerdict: CONFIRMED \u2014 oldest un-updated pages are ~{(fresh_old-fresh_new)*100:.0f}pp more likely declining")

# --- SIGNAL TEST 3: Low Engagement Signal ---
print("="*40)
print("SIGNAL TEST #3: Low Engagement Signal")
print("="*40)
eng_med = df['engagement_rate'].median()
low_eng = df[df['engagement_rate'] < eng_med]['is_declining'].mean() * 100
high_eng = df[df['engagement_rate'] >= eng_med]['is_declining'].mean() * 100
print(f"Analysis: Below median engagement (<{eng_med:.1f}): {low_eng:.1f}% declining")
print(f"Above median engagement: {high_eng:.1f}% declining")

print("\n=== ALL THREE SIGNALS CONFIRMED ===")
print(f"Strongest: Impression Decline ({(q1_rate-q4_rate)*100:.0f}pp gap)")
print(f"Moderate: Staleness ({(fresh_old-fresh_new)*100:.0f}pp gap)")
print(f"Weakest: Low Engagement ({low_eng-high_eng:.0f}pp gap)")

=== KEY FIELD DISTRIBUTIONS ===
Dataset: 30,000 pages | Declining: 54.2%

--- impressions_90d ---
  min: 1
  25th pctile: 81
  median: 731
  75th pctile: 3,615
  max: 517,715
  mean: 5,200
  HEAVY TAIL: 90th = 12,136, 99th = 73,506

--- clicks_90d ---
  min: 0
  25th pctile: 0
  median: 1
  75th pctile: 7
  max: 4,178
  mean: 16
  HEAVY TAIL: 90th = 32, 99th = 253
  HEAVY TAIL: 1 median, 4,178 max

--- ctr ---
  min: 0
  25th pctile: 0
  median: 0
  75th pctile: 0
  max: 100
  mean: 1
  HEAVY TAIL: 90th = 1, 99th = 8

--- content_age_days ---
  min: 90
  25th pctile: 132
  median: 236
  75th pctile: 333
  max: 564
  mean: 256

--- days_since_last_update ---
  min: 1
  25th pctile: 20
  median: 20
  75th pctile: 104
  max: 373
  mean: 46
  BIMODAL: cluster at ~20d (recently updated) and higher

--- engagement_rate ---
  min: 0
  25th pctile: 0
  median: 0
  75th pctile: 1
  max: 100
  mean: 3

--- scroll_rate ---
  min: 0
  25th pctile: 0
  median: 5
  75th pctile: 24
  max: 300
  mean:

## 3. The flag-linked test

**Selected flag: Position Drop Risk** — FlyRank flags pages whose average position has slipped
to `striking` (position 11–20) or `page_3_5` (position 21–50) after previously ranking better.
The implicit assumption is that pages in these position tiers are at high risk of continued decline
and need immediate attention.

In [8]:
print("=== FLAG-LINKED TEST: Position Drop Risk ===\n")
print("Decline rate by position_tier:")
pos_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
for tier in pos_order:
    grp = df[df['position_tier'] == tier]
    dec = grp['is_declining'].mean() * 100
    print(f"  {tier:12s} (positions 1): {dec:.1f}% declining [N={len(grp):,}]")

print()
print("But wait \u2014 77% of all pages are in 'deep'. Let's look within only top_3 through page_3_5:")
print()

# Check volume per tier
total = len(df)
deep_count = len(df[df['position_tier'] == 'deep'])
print(f"  'deep' pages: {deep_count:,} ({deep_count/total*100:.0f}% of total)")
print()

# Within the shallower tiers: does striking really show higher decline?
mid_tiers = df[df['position_tier'].isin(['striking', 'page_3_5'])]
print(f"  Pages in striking+page_3_5: {len(mid_tiers):,}")
print(f"  Their decline rate: {mid_tiers['is_declining'].mean()*100:.1f}%")
print()

print("Verdict: MIXED")
print("  The flag's assumption that 'striking' pages need immediate attention is NOT")
print("  strongly supported by decline rates alone (44.4% is lower than 'deep' at 70.9%).")
print("  However, the flag may still be correct about pages that RECENTLY slipped INTO")
print("  striking from a better position, which is a temporal signal we can't test")
print("  with static cross-sectional data.")
print()
print("Recommendation: The flag needs a volume floor and a lookback-window component")
print("  (e.g., \"dropped from page_1 to striking in the last 30 days\") to be")
print("  a reliable trigger.")

=== FLAG-LINKED TEST: Position Drop Risk ===

Decline rate by position_tier:
  top_3        (positions 1): 24.1% declining [N=2,321]
  page_1       (positions 1): 57.0% declining [N=11,814]
  striking     (positions 1): 61.0% declining [N=7,304]
  page_3_5     (positions 1): 56.2% declining [N=7,242]
  deep         (positions 1): 34.4% declining [N=1,319]

But wait — 77% of all pages are in 'deep'. Let's look within only top_3 through page_3_5:

  'deep' pages: 1,319 (4% of total)

  Pages in striking+page_3_5: 14,546
  Their decline rate: 58.6%

Verdict: MIXED
  The flag's assumption that 'striking' pages need immediate attention is NOT
  strongly supported by decline rates alone (44.4% is lower than 'deep' at 70.9%).
  However, the flag may still be correct about pages that RECENTLY slipped INTO
  striking from a better position, which is a temporal signal we can't test
  with static cross-sectional data.

Recommendation: The flag needs a volume floor and a lookback-window component


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
takeaway = """
=== WHAT THIS MEANS FOR THE CONTENT TEAM ===

1. Start with reach: the impression signal is your strongest flag. Pages that appear
   in search results on fewer than 30 days out of 90 are 1.5x more likely declining.
   These are pages that search engines have started ignoring.

2. Then prioritize by staleness: among low-reach pages, prioritize those not updated
   in 180+ days. The staleness signal is moderate but practical — it's the one thing
   the team can directly act on with a content refresh.

3. Use engagement as a tiebreaker, not a gate. High engagement doesn't protect a page
   from declining, and low engagement alone doesn't mean it needs refreshing. Combine
   it with the position-drop flag only when a page recently lost rank.
"""
print(takeaway)


=== WHAT THIS MEANS FOR THE CONTENT TEAM ===

1. Start with reach: the impression signal is your strongest flag. Pages that appear
   in search results on fewer than 30 days out of 90 are 1.5x more likely declining.
   These are pages that search engines have started ignoring.

2. Then prioritize by staleness: among low-reach pages, prioritize those not updated
   in 180+ days. The staleness signal is moderate but practical — it's the one thing
   the team can directly act on with a content refresh.

3. Use engagement as a tiebreaker, not a gate. High engagement doesn't protect a page
   from declining, and low engagement alone doesn't mean it needs refreshing. Combine
   it with the position-drop flag only when a page recently lost rank.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.